In [15]:
import random
from math import sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder


Implementação Seed

In [2]:
def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

# ErgoPose Risk Classifier — Model Training

This notebook is the **third stage** of the *ErgoPose Risk Classifier* project.  
It defines, trains, and evaluates the **Artificial Neural Network (ANN)** for posture classification based on the preprocessed dataset.

### Objectives
- Load the cleaned dataset from `data/processed/`.
- Encode categorical posture labels.
- Split the data into training and testing sets.
- Define and train an ANN model for multi-class classification.
- Save the trained model and scaler to the `models/` directory.

### Input and Output
- **Input:** `data/processed/clean_postural_risk_dataset.csv`  
- **Outputs:**  
  - `models/neural_network.pkl`  
  - `models/scaler.pkl`

In [3]:
PROJECT_ROOT = Path("..").resolve()

DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "clean_postural_risk_dataset.csv",
    Path("/content/drive/MyDrive/clean_postural_risk_dataset.csv"),
]

DATA_PATH = None
for cand in DATA_CANDIDATES:
    if cand.exists():
        DATA_PATH = cand
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Arquivo 'clean_postural_risk_dataset.csv' não encontrado em:\n"
        + "\n".join(str(p) for p in DATA_CANDIDATES)
    )

MODELS_PATH = PROJECT_ROOT / "models"
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Usando dataset em: {DATA_PATH}")
print(f"Modelos serão salvos em: {MODELS_PATH}")


Usando dataset em: C:\Faculdade\6 periodo\RNA\ergopose-risk-classifier\data\processed\clean_postural_risk_dataset.csv
Modelos serão salvos em: C:\Faculdade\6 periodo\RNA\ergopose-risk-classifier\models


In [4]:
data = pd.read_csv(DATA_PATH)

X = data.drop(columns=["upperbody_label"])
y = data["upperbody_label"]

print(f"X: {X.shape}")
print("distribuição de classe em 'upperbody_label':")
print(y.value_counts(normalize=True))

X: (4794, 51)
distribuição de classe em 'upperbody_label':
upperbody_label
bad     0.663121
good    0.336879
Name: proportion, dtype: float64


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


## Model Architecture Proposals - rules
- Hidden layers must have beetwen **5 and 20 neurons total**. If more than 1 hidden layer is implemented, the number of neurons of both layers must **add up** to a number beetwen **5 and 20**.
- Batch size, at this initial stage, must be **default**.
- Activation function **cannot** be tanh.
- Learning rate must be $10^{-3}$ or **smaller numbers**.

In [6]:
class MLPModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim_1: int, hidden_dim_2: int, output_dim: int):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),  # ativação não é tanh
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),  # ativação não é tanh
            nn.Linear(hidden_dim_2, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [7]:
def run_model(
    X_df: pd.DataFrame,
    y_series: pd.Series,
    input_dim: int,
    hidden_dim_1: int,
    hidden_dim_2: int,
    output_dim: int,
    num_epochs: int,
    learning_rate: float,
    n_splits: int,
    seed: int,
    device: torch.device,
):
    """
    Executa treino + validação cruzada KFold para uma dada combinação de hiperparâmetros
    e uma seed específica.

    Retorna:
        results: dict com médias de Acc, Precision, Recall, F1 nos folds.
        best_fold_model: modelo com melhor acurácia em um fold.
        best_fold_acc: valor de acurácia do melhor fold.
    """
    # Codifica rótulos em 0/1
    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y_series.values)

    X_np = X_df.values.astype(np.float32)
    y_np = y_enc.astype(np.float32)

    X_t = torch.from_numpy(X_np).to(device)
    y_t = torch.from_numpy(y_np).to(device)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1_scores = []

    best_fold_acc = 0.0
    best_fold_model = None

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_np, y_enc), start=1):
        X_train_t = X_t[train_idx]
        y_train_t = y_t[train_idx]
        X_val_t = X_t[val_idx]
        y_val_t = y_t[val_idx]

        model = MLPModel(
            input_dim=input_dim,
            hidden_dim_1=hidden_dim_1,
            hidden_dim_2=hidden_dim_2,
            output_dim=output_dim,
        ).to(device)

        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        for epoch in range(num_epochs):
            model.train()
            optimizer.zero_grad()
            outputs = model(X_train_t).squeeze()
            loss = criterion(outputs, y_train_t)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            outputs = model(X_val_t).squeeze()
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long().cpu().numpy()
            y_true = y_val_t.cpu().numpy().astype(int)

        acc = accuracy_score(y_true, preds)
        prec = precision_score(y_true, preds, zero_division=0)
        rec = recall_score(y_true, preds, zero_division=0)
        f1 = f1_score(y_true, preds, zero_division=0)

        fold_accuracies.append(acc)
        fold_precisions.append(prec)
        fold_recalls.append(rec)
        fold_f1_scores.append(f1)

        if acc > best_fold_acc:
            best_fold_acc = acc
            best_fold_model = model

        print(
            f"[Seed {seed}] Fold {fold_idx}/{n_splits} | "
            f"Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}",
        )

    results = {
        "Acc": float(np.mean(fold_accuracies)),
        "Precision": float(np.mean(fold_precisions)),
        "Recall": float(np.mean(fold_recalls)),
        "F1": float(np.mean(fold_f1_scores)),
    }

    print(
        f"[Seed {seed}] Médias nos {n_splits} folds → "
        f"Acc: {results['Acc']:.4f}, Prec: {results['Precision']:.4f}, "
        f"Rec: {results['Recall']:.4f}, F1: {results['F1']:.4f}",
    )

    return results, best_fold_model, best_fold_acc

In [8]:
input_dim = X.shape[1]
output_dim = 1
N_FOLDS = 5

NUM_SEEDS_PER_CONFIG = 10
BASE_SEED = 42

# 10 combinações diferentes de hiperparâmetros,
HYPERPARAM_CONFIGS = [
    {"name": "cfg_01", "hidden_dim_1": 3,  "hidden_dim_2": 2,  "learning_rate": 10e-3,  "num_epochs": 300},
    {"name": "cfg_02", "hidden_dim_1": 4,  "hidden_dim_2": 2,  "learning_rate": 5e-3,  "num_epochs": 300},
    {"name": "cfg_03", "hidden_dim_1": 4,  "hidden_dim_2": 3,  "learning_rate": 10e-3,  "num_epochs": 300},
    {"name": "cfg_04", "hidden_dim_1": 5,  "hidden_dim_2": 3,  "learning_rate": 5e-3,  "num_epochs": 350},
    {"name": "cfg_05", "hidden_dim_1": 5,  "hidden_dim_2": 4,  "learning_rate": 10e-4,  "num_epochs": 350},
    {"name": "cfg_06", "hidden_dim_1": 6,  "hidden_dim_2": 4,  "learning_rate": 5e-4,  "num_epochs": 400},
    {"name": "cfg_07", "hidden_dim_1": 7,  "hidden_dim_2": 4,  "learning_rate": 10e-3,  "num_epochs": 400},
    {"name": "cfg_08", "hidden_dim_1": 8,  "hidden_dim_2": 4,  "learning_rate": 5e-4,  "num_epochs": 450},
    {"name": "cfg_09", "hidden_dim_1": 8,  "hidden_dim_2": 5,  "learning_rate": 10e-4,  "num_epochs": 450},
    {"name": "cfg_10","hidden_dim_1": 10, "hidden_dim_2": 5,  "learning_rate": 5e-3,  "num_epochs": 500},
]


In [11]:
global_best_acc = 0.0
global_best_model_state = None
global_best_metadata = None

summary_rows = []

for cfg_idx, cfg in enumerate(HYPERPARAM_CONFIGS, start=1):
    print("\n" + "=" * 60)
    print(
        f"Config {cfg_idx}/{len(HYPERPARAM_CONFIGS)} → {cfg['name']} | "
        f"H1={cfg['hidden_dim_1']}, H2={cfg['hidden_dim_2']}, "
        f"lr={cfg['learning_rate']}, epochs={cfg['num_epochs']}"
    )
    print("=" * 60 + "\n")

    seed_metrics = []

    for seed_offset in range(NUM_SEEDS_PER_CONFIG):
        seed: int = BASE_SEED + seed_offset
        print(f"\n>>> Rodando {cfg['name']} com seed={seed}")
        set_seed(seed)

        results, best_model_seed, best_acc_seed = run_model(
            X_df=X,
            y_series=y,
            input_dim=input_dim,
            hidden_dim_1=cfg["hidden_dim_1"],
            hidden_dim_2=cfg["hidden_dim_2"],
            output_dim=output_dim,
            num_epochs=cfg["num_epochs"],
            learning_rate=cfg["learning_rate"],
            n_splits=N_FOLDS,
            seed=seed,
            device=device,
        )
        exp_name: str = f"{cfg['name']}_seed_{seed}"
        exp_path: Path = MODELS_PATH / exp_name
        exp_path.mkdir(parents=True, exist_ok=True)

        # salvar pesos do melhor modelo
        model_path = exp_path / "model.pth"
        torch.save(best_model_seed.state_dict(), model_path)

        # json hiperparâmetros + métricas
        metadata = {
            "experiment_name": exp_name,
            "config_name": cfg["name"],
            "seed": seed,
            "input_dim": input_dim,
            "output_dim": output_dim,
            "hidden_dim_1": cfg["hidden_dim_1"],
            "hidden_dim_2": cfg["hidden_dim_2"],
            "total_hidden_neurons": cfg["hidden_dim_1"] + cfg["hidden_dim_2"],
            "activation": "ReLU",
            "learning_rate": cfg["learning_rate"],
            "num_epochs": cfg["num_epochs"],
            "n_folds": N_FOLDS,
            "metrics": results,
        }

        metadata_path: Path = exp_path / "metadata.json"
        with metadata_path.open("w") as f:
            json.dump(metadata, f, indent=4)

        seed_metrics.append(results)

        # atualiza melhor modelo (usando melhor acc de fold desse seed)
        if best_acc_seed > global_best_acc:
            global_best_acc = best_acc_seed
            global_best_model_state = best_model_seed.state_dict()
            global_best_metadata = metadata

    # média de métricas dessa configuração ao longo das 10 seeds
    cfg_mean = {
        "config_name": cfg["name"],
        "hidden_dim_1": cfg["hidden_dim_1"],
        "hidden_dim_2": cfg["hidden_dim_2"],
        "total_hidden_neurons": cfg["hidden_dim_1"] + cfg["hidden_dim_2"],
        "learning_rate": cfg["learning_rate"],
        "num_epochs": cfg["num_epochs"],
    }
    for key in ["Acc", "Precision", "Recall", "F1"]:
        vals = [m[key] for m in seed_metrics]
        cfg_mean[f"mean_{key.lower()}"] = float(np.mean(vals))
        cfg_mean[f"std_{key.lower()}"] = float(np.std(vals))

    summary_rows.append(cfg_mean)


Config 1/10 → cfg_01 | H1=3, H2=2, lr=0.01, epochs=300


>>> Rodando cfg_01 com seed=42
[Seed 42] Fold 1/5 | Acc: 0.8655 | Prec: 0.8121 | Rec: 0.7846 | F1: 0.7981
[Seed 42] Fold 2/5 | Acc: 0.8717 | Prec: 0.8154 | Rec: 0.7814 | F1: 0.7980
[Seed 42] Fold 3/5 | Acc: 0.8697 | Prec: 0.7737 | Rec: 0.8829 | F1: 0.8247
[Seed 42] Fold 4/5 | Acc: 0.8332 | Prec: 0.7168 | Rec: 0.8515 | F1: 0.7784
[Seed 42] Fold 5/5 | Acc: 0.8643 | Prec: 0.7372 | Rec: 0.9146 | F1: 0.8164
[Seed 42] Médias nos 5 folds → Acc: 0.8609, Prec: 0.7711, Rec: 0.8430, F1: 0.8031

>>> Rodando cfg_01 com seed=43
[Seed 43] Fold 1/5 | Acc: 0.8926 | Prec: 0.7885 | Rec: 0.9321 | F1: 0.8543
[Seed 43] Fold 2/5 | Acc: 0.8561 | Prec: 0.7595 | Rec: 0.8515 | F1: 0.8029
[Seed 43] Fold 3/5 | Acc: 0.7842 | Prec: 0.7393 | Rec: 0.6070 | F1: 0.6667
[Seed 43] Fold 4/5 | Acc: 0.8686 | Prec: 0.7679 | Rec: 0.8431 | F1: 0.8037
[Seed 43] Fold 5/5 | Acc: 0.8758 | Prec: 0.8305 | Rec: 0.7803 | F1: 0.8046
[Seed 43] Médias nos 5 folds → Acc: 0.8554, Pre

In [12]:
summary_df = pd.DataFrame(summary_rows)
summary_csv_path: Path = MODELS_PATH / "hyperparam_search_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nresumo dos hiperparam e resultados salvos em: {summary_csv_path}")

if global_best_model_state is not None:
    best_model_path: Path = MODELS_PATH / "best_overall_model.pth"
    torch.save(global_best_model_state, best_model_path)

    best_metadata_path: Path = MODELS_PATH / "best_overall_model.json"
    with best_metadata_path.open("w") as f:
        json.dump(global_best_metadata, f, indent=4)

    print(
        f"\nmelhor modelo salvo em {best_model_path} "
        f"com melhor acc de fold = {global_best_acc:.4f}",
    )
    print(f"metadata do melhor modelo em: {best_metadata_path}")
else:
    print("nenhum modelo foi treinado com sucesso; nada salvo como melhor modelo.")


resumo dos hiperparam e resultados salvos em: C:\Faculdade\6 periodo\RNA\ergopose-risk-classifier\models\hyperparam_search_summary.csv

melhor modelo salvo em C:\Faculdade\6 periodo\RNA\ergopose-risk-classifier\models\best_overall_model.pth com melhor acc de fold = 0.9385
metadata do melhor modelo em: C:\Faculdade\6 periodo\RNA\ergopose-risk-classifier\models\best_overall_model.json
